## Iterable & Sequence Type
## Plain language

- An **iterable** is anything that can give you its items one at a time. It promises nothing else: maybe not how many items there are, maybe not a second pass, and no jumping straight to item 5.
- A **sequence** is an iterable that also keeps its items in fixed numbered positions. You can ask "how many?" (`len`), ask for "item 5" (`x[5]`), and walk through it as many times as you like.

Every sequence is iterable, but many iterables are not sequences.

## Analogy

- **Iterable = a baggage carousel.** Bags come past one at a time. You can't ask for bag #7 directly, you don't know how many are left, and once a bag is gone it's gone.
- **Sequence = a row of numbered lockers.** You know there are 50, you can open #7 directly, and you can walk the row again tomorrow and find the same things in the same places.

## The contracts

The "is it iterable?" check is defined by behaviour (what methods the object has). The "is it a sequence?" check is defined by declaration (the class has to say so). This is the part most people miss.

||**Iterable**|**Sequence**|
|---|---|---|
|Methods you must provide|`__iter__`|`__getitem__(int)` and `__len__`|
|Free extras from `collections.abc`|none|`__iter__`, `__contains__`, `__reversed__`, `index`, `count`|
|Works with `len(x)`|❌ not guaranteed|✅|
|Works with `x[i]` and slicing|❌|✅|
|Can be traversed many times|❌ not guaranteed|✅|
|How `isinstance` decides|**Duck typing**: it checks for `__iter__`|**Nominal**: the class must inherit from `Sequence` or be registered|
|Examples|generator, file, `set`, `dict`, `map`, `zip`|`list`, `tuple`, `str`, `range`, `bytes`|

The hierarchy in `collections.abc`:

```
Iterable ─┐
Sized ────┼─► Collection ─► Sequence ─► MutableSequence
Container ┘       │            (+ Reversible)    (list)
                  ├─► Set ─► MutableSet           (set)
                  └─► Mapping ─► MutableMapping   (dict)
```

`dict` and `set` are **Collections**, meaning they support `len`, `in` and iteration. They are not Sequences, because they have no integer positions.

### Minimal example

In [ ]:
from collections.abc import Iterable, Sequence


def countdown(start: int):
    yield from range(start, 0, -1)

gen = countdown(3)
nums = [3, 2, 1]

isinstance(gen, Iterable), isinstance(gen, Sequence)     # (True, False)
isinstance(nums, Iterable), isinstance(nums, Sequence)   # (True, True)
isinstance({1, 2}, Sequence)                             # False: no positions
isinstance({"a": 1}, Sequence)                           # False, even though dicts keep insertion order

len(nums), nums[0], nums[-1]   # (3, 3, 1)
len(gen)                       # TypeError: object of type 'generator' has no len()
gen[0]                         # TypeError: 'generator' object is not subscriptable

list(gen), list(gen)           # ([3, 2, 1], [])   one pass only
list(nums), list(nums)         # ([3, 2, 1], [3, 2, 1])

### Building your own Sequence
Write the two required methods and `Sequence` fills in the rest:

In [2]:
from collections.abc import Sequence
from typing import overload


class Page(Sequence[str]):
    """A read-only page of results. Inherits index, count, `in`, reversed()."""

    def __init__(self, items: list[str]) -> None:
        self._items = items

    @overload
    def __getitem__(self, index: int) -> str: ...
    @overload
    def __getitem__(self, index: slice) -> Sequence[str]: ...
    def __getitem__(self, index: int | slice) -> str | Sequence[str]:
        return self._items[index]

    def __len__(self) -> int:
        return len(self._items)

In [3]:
page = Page(["alice", "bob", "carol"])
"bob" in page            # True: mixin __contains__

True

In [4]:
page.index("carol")      # 2: mixin index()

2

In [5]:
list(reversed(page))     # ['carol', 'bob', 'alice']: mixin __reversed__

['carol', 'bob', 'alice']

How the free `__iter__` works: it calls `self[0]`, `self[1]`, and so on until it gets an `IndexError`. That is why a sequence never needs to write `__iter__` itself.

### Where you've already seen this

**Function signatures.** This is the decision you'll make most often:

In [ ]:
from collections.abc import Iterable, Sequence

# Only loops once → accept the weakest contract.
# A generator, a QuerySet, a set or a list will all work.
def bulk_send_emails(addresses: Iterable[str]) -> int:
    sent = 0
    for address in addresses:
        send(address)
        sent += 1
    return sent

# Needs len() and indexing → ask for a Sequence.
def paginate(items: Sequence[Order], page: int, size: int = 20) -> Sequence[Order]:
    total_pages = -(-len(items) // size)   # ceiling division needs len()
    if page > total_pages:
        raise ValueError(f"{page = } exceeds {total_pages = }")
    return items[(page - 1) * size : page * size]

The rule is to accept the most general type you can and return the most specific type you have. If `bulk_send_emails` asked for `list[str]`, callers would have to write `list(queryset)` and load every row into memory for no reason.

**Django QuerySet** is a good real-world edge case. It supports `len(qs)`, `qs[5]` and `qs[:10]`, so it looks like a sequence. But:

In [ ]:
isinstance(qs, Sequence)   # False: never registered
qs[-1]                     # ValueError: Negative indexing is not supported.

It behaves _mostly_ like a sequence, but it deliberately doesn't claim to be one. `qs[-1]` would need the database to count all rows first, so Django refuses.

## Gotchas an interviewer might ask about

**1. "Is a class with only `__getitem__` iterable?"**

Yes, through an older fallback protocol. But `isinstance` says no:

In [ ]:
class Legacy:
    def __getitem__(self, index: int) -> int:
        if index >= 3:
            raise IndexError
        return index * 10

list(Legacy())                       # [0, 10, 20]: iter() falls back to __getitem__
isinstance(Legacy(), Iterable)       # False: the ABC only looks for __iter__

So the only fully reliable "is it iterable?" check is to try `iter(obj)` and catch `TypeError`.

**2. "Is a class with `__getitem__` and `__len__` a Sequence?"**

Not according to `isinstance`. `Iterable` has a `__subclasshook__` that duck-types on `__iter__`. `Sequence` has no such hook, so you must inherit from it or call `Sequence.register(MyClass)`. The reason: `__getitem__` + `__len__` is also exactly what a dict has, so those two methods alone can't tell you whether the keys are integer positions.

**3. "`str` is a Sequence of `str`."**

In [ ]:
def tag_all(tags: Sequence[str]) -> None:
    for tag in tags:
        ...

tag_all("python")   # passes the type checker, then loops over 'p', 'y', 't', ...

This is a common real bug. Guard against it with `if isinstance(tags, str): raise TypeError(...)`.

**4. "Iterating an Iterable twice."**

In [ ]:
def stats(values: Iterable[float]) -> tuple[float, float]:
    total = sum(values)
    count = sum(1 for _ in values)   # 0 if values was a generator
    return total, total / count      # ZeroDivisionError

If your function needs two passes, either ask for a `Sequence` or call `values = list(values)` once at the top.

### Interview-ready summary

> "An iterable can hand you items one at a time. That's all it promises. A sequence adds `len`, integer indexing and repeatable traversal, so every sequence is iterable but not the reverse: **sets, dicts** and **generators** aren't sequences. One subtlety: `isinstance(x, Iterable)` is duck-typed on `__iter__`, while `Sequence` is nominal, so a class must inherit from it or be registered. For parameters I accept `Iterable` if I loop once and `Sequence` if I need `len`, indexing or multiple passes."

### Try this on your own

Without running it, predict each result, then check in a REPL:

In [ ]:
from collections.abc import Sequence, Collection, Reversible

isinstance(range(10), Sequence)
isinstance({}.keys(), Collection)
isinstance({}.keys(), Sequence)
isinstance(reversed([1, 2]), Sequence)
isinstance({"a": 1}, Reversible)   # hint: Python 3.8+

If you can explain why each answer is True or False, you've got this topic. This also sets up Protocol vs ABC (structural vs nominal typing) nicely if you want to cover that next.